# ⚡ Módulo 13 - Notebook 04: Delta Lake ACID y Time Travel

## 🔒 Almacenamiento Transaccional y Versionado de Datos

**Libro:** Saliendo de lo Pandito  
**Módulo:** 13 - PySpark SQL Window DeltaLake  
**Duración estimada:** 80 minutos  
**Dificultad:** 🔴 Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Crear** tablas Delta Lake con transacciones ACID  
✅ **Aplicar** Time Travel para acceder a versiones  
✅ **Ejecutar** Merge/Upsert para CDC  
✅ **Optimizar** tablas Delta (OPTIMIZE, VACUUM)  
✅ **Auditar** cambios con historial de transacciones

---

## 📋 Pre-requisitos

* ✅ Módulo 13 (notebooks 01-03) completado
* ✅ Conocimiento de PySpark SQL
* ✅ Familiaridad con conceptos de transacciones

---

## 📚 Contenido

1. ¿Qué es Delta Lake?
2. Transacciones ACID
3. Time Travel y Versionado
4. Merge/Upsert para CDC
5. OPTIMIZE y Z-Ordering
6. Caso Integrador: Pipeline Delta Completo

---

## 💡 Por qué importa

**Delta Lake revoluciona data lakes:**

* 🔒 **ACID:** Transacciones confiables
* 🕐 **Time Travel:** Auditoría y rollback
* 🔄 **Merge:** ETL incremental eficiente
* ⚡ **Performance:** Optimizaciones automáticas

**El formato de producción de facto en Databricks**

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from delta.tables import *

print("💾 PREPARANDO DATOS PARA DELTA LAKE")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market (SPARK DataFrame)
    df_source = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros: {df_source.count():,}")
    print(f"   📅 Período: {df_source.select(F.min('fecha'), F.max('fecha')).collect()[0]}")
    print(f"   📍 Ubicación: Mendoza, Argentina (Los Andes Market)")
    
    # Crear tabla Delta de ejemplo
    DELTA_PATH = "/tmp/delta_ventas_demo"
    
    print(f"\n🔷 Tabla Delta de ejemplo:")
    print(f"   Ruta: {DELTA_PATH}")
    print(f"   Formato: Delta Lake (ACID)")
    
    # Escribir como Delta
    df_source.write \
        .format("delta") \
        .mode("overwrite") \
        .save(DELTA_PATH)
    
    print(f"\n✅ Tabla Delta creada exitosamente")
    
    # Leer la tabla Delta
    df_delta = spark.read.format("delta").load(DELTA_PATH)
    print(f"   📊 Registros en Delta: {df_delta.count():,}")
    
    print(f"\n🎯 Este notebook demostrará:")
    print(f"   • Transacciones ACID")
    print(f"   • Time Travel (versiones)")
    print(f"   • Merge/Upsert")
    print(f"   • OPTIMIZE y VACUUM")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df_source = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Delta Lake: El Formato del Futuro

### 🔷 ¿Qué es Delta Lake?

**Delta Lake** es un formato de almacenamiento de código abierto que agrega capacidades ACID sobre Parquet.

**Arquitectura:**
```
Delta Lake = Parquet + Transaction Log
```

**Transaction Log:**
* Archivo JSON que registra cada cambio
* Ubicado en `_delta_log/`
* Permite ACID y Time Travel

---

### 🔒 Transacciones ACID

**ACID:**

**A**tomicity: Todo o nada
* Si falla la escritura, NO hay corrupción

**C**onsistency: Datos consistentes
* Siempre ves una versión válida

**I**solation: Sin interferencias
* Lectores no bloquean escritores

**D**urability: Cambios permanentes
* Una vez committed, no se pierde

**Ventaja vs Parquet:**
* Parquet: Sin transacciones (puede corromperse)
* Delta: Transacciones garantizadas

---

### 🕐 Time Travel

**Acceder a versiones anteriores de una tabla:**

**Por número de versión:**
```python
df = spark.read \
    .format("delta") \
    .option("versionAsOf", 5) \
    .load("/path/to/delta")
```

**Por timestamp:**
```python
df = spark.read \
    .format("delta") \
    .option("timestampAsOf", "2024-01-01 10:00:00") \
    .load("/path/to/delta")
```

**Ver historial:**
```python
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, "/path/to/delta")
delta_table.history().show()
```

---

### 🔄 Merge/Upsert para CDC

**CDC** (Change Data Capture): Capturar cambios incrementales.

**Problema:** Actualizar registros existentes + insertar nuevos.

**Solución con Merge:**
```python
from delta.tables import DeltaTable

# Target: Tabla Delta existente
target = DeltaTable.forPath(spark, "/path/to/delta")

# Source: Nuevos datos
source = spark.table("nuevos_datos")

# Merge
target.alias("t").merge(
    source.alias("s"),
    "t.id = s.id"
).whenMatchedUpdate(
    set = {"valor": "s.valor", "fecha_update": "current_timestamp()"}
).whenNotMatchedInsert(
    values = {"id": "s.id", "valor": "s.valor", "fecha_insert": "current_timestamp()"}
).execute()
```

**Resultado:** Updates + Inserts en una sola operación atómica.

---

### ⚡ OPTIMIZE: Compactación

**Problema:** Escrituras frecuentes crean muchos archivos pequeños.

**Solución:**
```sql
OPTIMIZE delta_table
```

**Z-Ordering:** Agrupar datos relacionados
```sql
OPTIMIZE delta_table
ZORDER BY (columna_frecuente)
```

**Uso:** Columnas usadas frecuentemente en filtros (WHERE).

---

### 🗑️ VACUUM: Limpieza

**Problema:** Time Travel mantiene versiones antiguas (consume espacio).

**Solución:**
```sql
VACUUM delta_table RETAIN 168 HOURS  -- 7 días
```

**Efecto:** Elimina archivos de versiones > 7 días.

⚠️ **Cuidado:** Después de VACUUM no puedes hacer Time Travel a versiones eliminadas.

---

### 📊 Comparativa: Parquet vs Delta

| Característica | Parquet | Delta Lake |
|----------------|---------|------------|
| **Formato** | Columnar | Parquet + Log |
| **ACID** | ❌ No | ✅ Sí |
| **Time Travel** | ❌ No | ✅ Sí |
| **Update/Delete** | ❌ Sobrescritura | ✅ Atómico |
| **Merge** | ❌ No | ✅ Sí |
| **Schema Evolution** | Manual | Automático |
| **Optimización** | Manual | OPTIMIZE |

---

### 💼 Caso de Uso: Pipeline Incremental

**Escenario:** Actualizar ventas diarias.

**Sin Delta (Parquet):**
```python
# Problema: Sobrescribe TODO
df.write.mode("overwrite").parquet("/path")
```

**Con Delta:**
```python
# 1. Carga inicial
df_inicial.write.format("delta").save("/path")

# 2. Carga incremental (solo nuevos datos)
df_nuevos.write.format("delta").mode("append").save("/path")

# 3. Actualización (Merge)
target.merge(source, "t.id = s.id") \
    .whenMatchedUpdate(...) \
    .whenNotMatchedInsert(...) \
    .execute()
```

**Ventaja:** Solo procesas cambios (10x más rápido).

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
import warnings
warnings.filterwarnings('ignore')

print("🔷 DELTA LAKE: ACID Y TIME TRAVEL")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

try:
    print(f"Versión de Spark: {spark.version}")
except:
    print("⚠️  SparkSession no disponible")

print("\n🎯 En este notebook aprenderás:")
print("  • df.write.format('delta').save(path)")
print("  • spark.read.format('delta').load(path)")
print("  • Time Travel - option('versionAsOf', n)")
print("  • DeltaTable.forPath().merge()")
print("  • OPTIMIZE y VACUUM")

print("\n📖 Métodos clave:")
print("  - df.write.format('delta').mode('overwrite').save(path)")
print("  - spark.read.format('delta').option('versionAsOf', 5).load(path)")
print("  - DeltaTable.forPath(spark, path)")
print("  - delta_table.merge(source, condition)")
print("  - delta_table.history().show()")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

In [0]:
# Código de inicialización de notebook reindexado
import pandas as pd
import numpy as np
print('Notebook reindexado listo para práctica en Databricks')